In [1]:
import math
import copy
import os
import re
import glob
import sys
import subprocess
import random
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
)


SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CLASS_NAMES = ["Normal", "DoS", "DDoS", "Probe", "Botnet",
               "BruteForce", "WebAttack", "Infiltration"]
NORMAL_LABEL = 0
ATTACK_FAMILY = {
    "Normal": "benign",
    "DoS": "volumetric",
    "DDoS": "volumetric",
    "Probe": "recon",
    "Botnet": "c2",
    "BruteForce": "credential",
    "WebAttack": "webapp",
    "Infiltration": "intrusion",
}
PROTO_NAMES = ["TCP", "UDP", "ICMP", "OTHER"]


def set_taxonomy(names, families, normal_label=0):
    global CLASS_NAMES, ATTACK_FAMILY, NORMAL_LABEL
    CLASS_NAMES = list(names)
    ATTACK_FAMILY = dict(families)
    NORMAL_LABEL = normal_label


@dataclass
class Config:
    dataset: str = "unsw"
    unsw_slug: str = "mrwellsdavid/unsw-nb15"
    unsw_max_rows: int = 0
    max_windows: int = 80000
    balance_majority: bool = True
    majority_multiple: float = 3.0
    minority_floor: int = 300

    n_hosts: int = 220
    n_services: int = 6
    flows_per_host: int = 90
    raw_dim: int = 40
    nan_ratio: float = 0.01
    mi_keep: int = 26
    corr_thresh: float = 0.96
    seq_len: int = 16
    seq_stride: int = 16
    n_port_buckets: int = 16
    n_protocols: int = 16

    d_model: int = 64
    n_heads: int = 4
    n_transformer_layers: int = 2
    ff_dim: int = 128
    n_lstm_layers: int = 2
    lstm_hidden: int = 64
    dropout: float = 0.1
    svdd_dim: int = 64
    proj_dim: int = 64
    ctx_emb_dim: int = 32

    batch_size: int = 128
    pretrain_epochs: int = 6
    supervised_epochs: int = 12
    lr: float = 1.0e-3
    weight_decay: float = 1.0e-4
    mask_ratio: float = 0.15
    curriculum_warmup: int = 4
    ntxent_temp: float = 0.5
    lambda_mtm: float = 1.0
    lambda_contrast: float = 0.5
    lambda_svdd: float = 0.1
    focal_gamma: float = 2.0
    svdd_quantile: float = 0.95

    ewc_lambda: float = 30.0
    replay_capacity: int = 2000
    replay_per_batch: int = 32

    use_transformer: bool = True
    use_lstm: bool = True
    use_phi: bool = True
    use_ssl: bool = True
    vector_gate: bool = True


def _class_profile(num_classes, raw_dim):
    rng = np.random.RandomState(7)
    means = rng.uniform(-1.0, 1.0, size=(num_classes, raw_dim))
    scales = rng.uniform(0.4, 1.2, size=(num_classes, raw_dim))
    for c in range(num_classes):
        active = rng.choice(raw_dim, size=raw_dim // 3, replace=False)
        means[c, active] += rng.uniform(1.5, 4.0, size=active.shape) * rng.choice([-1, 1], size=active.shape)
    return means, scales


def _port_distribution(num_classes, n_buckets):
    rng = np.random.RandomState(11)
    dist = rng.uniform(0.2, 1.0, size=(num_classes, n_buckets))
    for c in range(num_classes):
        peak = rng.choice(n_buckets, size=2, replace=False)
        dist[c, peak] += rng.uniform(3.0, 6.0, size=peak.shape)
    return dist / dist.sum(axis=1, keepdims=True)


def _proto_distribution(num_classes, n_proto):
    rng = np.random.RandomState(13)
    dist = rng.uniform(0.2, 1.0, size=(num_classes, n_proto))
    return dist / dist.sum(axis=1, keepdims=True)


def generate_flows(cfg):
    num_classes = len(CLASS_NAMES)
    means, scales = _class_profile(num_classes, cfg.raw_dim)
    port_dist = _port_distribution(num_classes, cfg.n_port_buckets)
    proto_dist = _proto_distribution(num_classes, cfg.n_protocols)
    rng = np.random.RandomState(SEED)

    feats, labels, hosts, services, ports, protos, times = [], [], [], [], [], [], []
    for h in range(cfg.n_hosts):
        svc = rng.randint(0, cfg.n_services)
        n = cfg.flows_per_host
        prev = np.zeros(cfg.raw_dim)
        t = 0.0
        i = 0
        while i < n:
            if rng.rand() < 0.30:
                cls = rng.randint(1, num_classes)
                ep_len = rng.randint(4, 14)
            else:
                cls = NORMAL_LABEL
                ep_len = rng.randint(6, 18)
            for _ in range(ep_len):
                if i >= n:
                    break
                noise = rng.randn(cfg.raw_dim) * scales[cls]
                x = 0.6 * prev + 0.4 * (means[cls] + noise)
                prev = x
                pb = rng.choice(cfg.n_port_buckets, p=port_dist[cls])
                pr = rng.choice(cfg.n_protocols, p=proto_dist[cls])
                t += rng.exponential(1.0)
                feats.append(x)
                labels.append(cls)
                hosts.append(h)
                services.append(svc)
                ports.append(pb)
                protos.append(pr)
                times.append(t)
                i += 1

    X = np.asarray(feats, dtype=np.float64)
    y = np.asarray(labels, dtype=np.int64)
    host = np.asarray(hosts, dtype=np.int64)
    service = np.asarray(services, dtype=np.int64)
    port = np.asarray(ports, dtype=np.int64)
    proto = np.asarray(protos, dtype=np.int64)
    ts = np.asarray(times, dtype=np.float64)

    n_nan = int(cfg.nan_ratio * X.size)
    ri = rng.randint(0, X.shape[0], size=n_nan)
    ci = rng.randint(0, X.shape[1], size=n_nan)
    X[ri, ci] = np.nan
    return dict(X=X, y=y, host=host, service=service, port=port, proto=proto, ts=ts)


def impute_and_clip(X, lo=None, hi=None, med=None):
    if med is None:
        med = np.nanmedian(X, axis=0)
    inds = np.where(np.isnan(X))
    X = X.copy()
    X[inds] = np.take(med, inds[1])
    if lo is None or hi is None:
        lo = np.percentile(X, 1, axis=0)
        hi = np.percentile(X, 99, axis=0)
    X = np.clip(X, lo, hi)
    return X, lo, hi, med


def select_features(X, y, keep, corr_thresh):
    mi = mutual_info_classif(X, y, random_state=SEED)
    order = np.argsort(mi)[::-1]
    chosen = []
    for idx in order:
        if len(chosen) >= keep:
            break
        ok = True
        for c in chosen:
            cc = np.corrcoef(X[:, idx], X[:, c])[0, 1]
            if abs(cc) > corr_thresh:
                ok = False
                break
        if ok:
            chosen.append(idx)
    return np.asarray(sorted(chosen), dtype=np.int64)


def build_sequences(data, feat_idx, scaler, cfg, host_set):
    X = data["X"][:, feat_idx]
    X = scaler.transform(X)
    y, host, service = data["y"], data["host"], data["service"]
    port, proto, ts = data["port"], data["proto"], data["ts"]

    seqs, sports, sprotos, slabels = [], [], [], []
    keys = {}
    for i in range(len(y)):
        if host[i] not in host_set:
            continue
        k = (host[i], service[i])
        keys.setdefault(k, []).append(i)

    for k, idxs in keys.items():
        idxs = sorted(idxs, key=lambda j: ts[j])
        if len(idxs) < cfg.seq_len:
            continue
        for s in range(0, len(idxs) - cfg.seq_len + 1, cfg.seq_stride):
            win = idxs[s:s + cfg.seq_len]
            seqs.append(X[win])
            sports.append(port[win])
            sprotos.append(proto[win])
            slabels.append(y[win[-1]])

    Xs = torch.tensor(np.asarray(seqs), dtype=torch.float32)
    Ps = torch.tensor(np.asarray(sports), dtype=torch.long)
    Rs = torch.tensor(np.asarray(sprotos), dtype=torch.long)
    Ys = torch.tensor(np.asarray(slabels), dtype=torch.long)
    return Xs, Ps, Rs, Ys


def make_splits(cfg):
    data = generate_flows(cfg)
    rng = np.random.RandomState(SEED)
    hosts = np.arange(cfg.n_hosts)
    rng.shuffle(hosts)
    n_tr = int(0.6 * cfg.n_hosts)
    n_va = int(0.2 * cfg.n_hosts)
    tr_h = set(hosts[:n_tr].tolist())
    va_h = set(hosts[n_tr:n_tr + n_va].tolist())
    te_h = set(hosts[n_tr + n_va:].tolist())

    Xtr_flat = data["X"][np.isin(data["host"], list(tr_h))]
    ytr_flat = data["y"][np.isin(data["host"], list(tr_h))]
    Xtr_flat, lo, hi, med = impute_and_clip(Xtr_flat)

    feat_idx = select_features(Xtr_flat, ytr_flat, cfg.mi_keep, cfg.corr_thresh)
    scaler = RobustScaler().fit(Xtr_flat[:, feat_idx])

    data["X"], _, _, _ = impute_and_clip(data["X"], lo, hi, med)

    train = build_sequences(data, feat_idx, scaler, cfg, tr_h)
    val = build_sequences(data, feat_idx, scaler, cfg, va_h)
    test = build_sequences(data, feat_idx, scaler, cfg, te_h)
    return train, val, test, feat_idx, scaler


UNSW_COLS = [
    "srcip", "sport", "dstip", "dsport", "proto", "state", "dur", "sbytes", "dbytes",
    "sttl", "dttl", "sloss", "dloss", "service", "sload", "dload", "spkts", "dpkts",
    "swin", "dwin", "stcpb", "dtcpb", "smeansz", "dmeansz", "trans_depth", "res_bdy_len",
    "sjit", "djit", "stime", "ltime", "sintpkt", "dintpkt", "tcprtt", "synack", "ackdat",
    "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd",
    "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm",
    "ct_dst_sport_ltm", "ct_dst_src_ltm", "attack_cat", "label",
]
UNSW_NUMERIC = [
    "dur", "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "sload", "dload",
    "spkts", "dpkts", "swin", "dwin", "stcpb", "dtcpb", "smeansz", "dmeansz",
    "trans_depth", "res_bdy_len", "sjit", "djit", "sintpkt", "dintpkt", "tcprtt",
    "synack", "ackdat", "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd",
    "is_ftp_login", "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm",
    "ct_src_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm",
]
UNSW_CLASS_NAMES = ["Normal", "Fuzzers", "Analysis", "Backdoor", "DoS", "Exploits",
                    "Generic", "Reconnaissance", "Shellcode", "Worms"]
UNSW_FAMILY = {n: ("benign" if n == "Normal" else n.lower()) for n in UNSW_CLASS_NAMES}
UNSW_ATTACK_ALIASES = {
    "backdoors": "Backdoor", "backdoor": "Backdoor", "fuzzers": "Fuzzers",
    "analysis": "Analysis", "dos": "DoS", "exploits": "Exploits", "generic": "Generic",
    "reconnaissance": "Reconnaissance", "shellcode": "Shellcode", "worms": "Worms",
}


def _ensure_kagglehub():
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    return kagglehub


def _read_unsw_csv(fp):
    head = pd.read_csv(fp, nrows=3, header=None, dtype=str, encoding="latin-1")
    first = [str(v).strip().lower() for v in head.iloc[0].tolist()]
    has_header = "srcip" in first or "proto" in first and "service" in first
    df = pd.read_csv(fp, header=0 if has_header else None, dtype=str,
                     low_memory=False, encoding="latin-1")
    if not has_header:
        n = df.shape[1]
        df.columns = UNSW_COLS[:n] if n <= len(UNSW_COLS) else list(range(n))
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def _safe_port(v):
    try:
        return int(float(v))
    except (ValueError, TypeError):
        try:
            return int(str(v).strip(), 16)
        except (ValueError, TypeError):
            return -1


def _bucketize_ports(ports, nb):
    p = np.clip(ports, 0, 65535).astype(np.float64)
    b = np.floor(p / 65536.0 * (nb - 1)).astype(np.int64)
    b[ports < 0] = nb - 1
    return np.clip(b, 0, nb - 1)


def _map_proto(series, nb):
    s = series.astype(str).str.strip().str.lower()
    top = list(s.value_counts().index[: nb - 1])
    mapping = {name: i for i, name in enumerate(top)}
    codes = s.map(mapping).fillna(nb - 1).astype(np.int64).values
    return codes


def load_unsw_nb15(cfg):
    set_taxonomy(UNSW_CLASS_NAMES, UNSW_FAMILY, normal_label=0)
    kagglehub = _ensure_kagglehub()
    path = kagglehub.dataset_download(cfg.unsw_slug)
    all_csv = []
    for root, _, files in os.walk(path):
        for f in files:
            if f.lower().endswith(".csv"):
                all_csv.append(os.path.join(root, f))
    data_files = sorted([f for f in all_csv
                         if re.search(r"nb15_[1-4]\.csv$", os.path.basename(f).lower())])
    if not data_files:
        raise FileNotFoundError(
            "UNSW-NB15_1..4.csv not found under " + path +
            ". Use a kagglehub dataset that contains the full four CSV files.")
    frames = []
    for fp in data_files:
        df = _read_unsw_csv(fp)
        frames.append(df)
        if cfg.unsw_max_rows and sum(len(x) for x in frames) >= cfg.unsw_max_rows:
            break
    df = pd.concat(frames, ignore_index=True)
    if cfg.unsw_max_rows and len(df) > cfg.unsw_max_rows:
        df = df.iloc[: cfg.unsw_max_rows].copy()

    name_to_idx = {n: i for i, n in enumerate(UNSW_CLASS_NAMES)}
    ac = df["attack_cat"].astype(str).str.strip()
    ac = ac.replace({"nan": "", "-": "", "None": ""})
    ac_norm = ac.str.lower().map(UNSW_ATTACK_ALIASES)
    y = ac_norm.map(name_to_idx)
    y = y.where(ac.str.len() > 0, 0).fillna(0).astype(np.int64).values

    Xdf = df[UNSW_NUMERIC].apply(pd.to_numeric, errors="coerce")
    Xdf = Xdf.replace([np.inf, -np.inf], np.nan)
    state_code = pd.Series(pd.factorize(df["state"].astype(str).str.strip())[0],
                           name="state_code")
    X = np.column_stack([Xdf.values.astype(np.float64),
                         state_code.values.astype(np.float64)])

    host = pd.factorize(df["srcip"].astype(str))[0].astype(np.int64)
    service = pd.factorize(df["service"].astype(str).str.strip())[0].astype(np.int64)
    ports = np.array([_safe_port(v) for v in df["dsport"].values], dtype=np.int64)
    port = _bucketize_ports(ports, cfg.n_port_buckets)
    proto = _map_proto(df["proto"], cfg.n_protocols)
    ts = pd.to_numeric(df["stime"], errors="coerce").ffill().fillna(0).values
    if np.allclose(ts, ts[0]):
        ts = np.arange(len(df), dtype=np.float64)

    return dict(X=X, y=y, host=host, service=service, port=port, proto=proto, ts=ts)


def build_window_indices(data, cfg):
    y, host, service, ts = data["y"], data["host"], data["service"], data["ts"]
    groups = {}
    for i in range(len(y)):
        groups.setdefault((int(host[i]), int(service[i])), []).append(i)
    windows = []
    for _, idxs in groups.items():
        idxs.sort(key=lambda j: ts[j])
        for s in range(0, len(idxs) - cfg.seq_len + 1, cfg.seq_stride):
            win = idxs[s:s + cfg.seq_len]
            windows.append((np.asarray(win, dtype=np.int64), int(y[win[-1]])))
    return windows


def stratified_split_windows(windows, cfg):
    rng = np.random.RandomState(SEED)
    by_cls = {}
    for i, (_, lab) in enumerate(windows):
        by_cls.setdefault(lab, []).append(i)
    tr, va, te = [], [], []
    for lab, idxs in by_cls.items():
        idxs = np.asarray(idxs)
        rng.shuffle(idxs)
        n = len(idxs)
        a, b = int(0.6 * n), int(0.8 * n)
        tr += idxs[:a].tolist()
        va += idxs[a:b].tolist()
        te += idxs[b:].tolist()
    if cfg.max_windows and len(tr) > int(0.6 * cfg.max_windows):
        rng.shuffle(tr)
        tr = tr[: int(0.6 * cfg.max_windows)]
        rng.shuffle(va)
        va = va[: int(0.2 * cfg.max_windows)]
        rng.shuffle(te)
        te = te[: int(0.2 * cfg.max_windows)]
    return tr, va, te


def assemble_windows(data, windows, idx_list, feat_idx, scaler):
    Xn = scaler.transform(data["X"][:, feat_idx]).astype(np.float32)
    port, proto = data["port"], data["proto"]
    Xs, Ps, Rs, Ys = [], [], [], []
    for i in idx_list:
        win, lab = windows[i]
        Xs.append(Xn[win])
        Ps.append(port[win])
        Rs.append(proto[win])
        Ys.append(lab)
    return (torch.tensor(np.asarray(Xs), dtype=torch.float32),
            torch.tensor(np.asarray(Ps), dtype=torch.long),
            torch.tensor(np.asarray(Rs), dtype=torch.long),
            torch.tensor(np.asarray(Ys), dtype=torch.long))


def balance_train_indices(idx_list, windows, cfg):
    if not cfg.balance_majority:
        return idx_list
    rng = np.random.RandomState(SEED)
    idx_arr = np.asarray(idx_list)
    labels = np.asarray([windows[i][1] for i in idx_list])
    counts = np.bincount(labels, minlength=len(CLASS_NAMES))
    attack_max = counts[1:].max() if counts[1:].size and counts[1:].max() > 0 else counts.max()
    cap = int(max(attack_max * cfg.majority_multiple, cfg.minority_floor))
    keep = []
    for c in np.unique(labels):
        ci = idx_arr[labels == c]
        if c == NORMAL_LABEL and len(ci) > cap:
            ci = rng.choice(ci, cap, replace=False)
        elif len(ci) < cfg.minority_floor:
            reps = int(np.ceil(cfg.minority_floor / len(ci)))
            ci = np.tile(ci, reps)[: cfg.minority_floor]
        keep.extend(ci.tolist())
    rng.shuffle(keep)
    return keep


def make_splits_unsw(cfg):
    data = load_unsw_nb15(cfg)
    windows = build_window_indices(data, cfg)
    tr, va, te = stratified_split_windows(windows, cfg)
    tr = balance_train_indices(tr, windows, cfg)

    train_flows = np.unique(np.concatenate([windows[i][0] for i in tr]))
    Xtr_flat, lo, hi, med = impute_and_clip(data["X"][train_flows])
    ytr_flat = data["y"][train_flows]
    if len(Xtr_flat) > 60000:
        sub = np.random.RandomState(SEED).choice(len(Xtr_flat), 60000, replace=False)
        feat_idx = select_features(Xtr_flat[sub], ytr_flat[sub], cfg.mi_keep, cfg.corr_thresh)
    else:
        feat_idx = select_features(Xtr_flat, ytr_flat, cfg.mi_keep, cfg.corr_thresh)
    scaler = RobustScaler().fit(Xtr_flat[:, feat_idx])
    data["X"], _, _, _ = impute_and_clip(data["X"], lo, hi, med)

    train = assemble_windows(data, windows, tr, feat_idx, scaler)
    val = assemble_windows(data, windows, va, feat_idx, scaler)
    test = assemble_windows(data, windows, te, feat_idx, scaler)
    return train, val, test, feat_idx, scaler


def temporal_jitter(x, p, r):
    noise = torch.randn_like(x) * 0.08
    perm = x.clone()
    T = x.size(1)
    if T > 1:
        for b in range(x.size(0)):
            i = random.randint(0, T - 2)
            perm[b, i], perm[b, i + 1] = x[b, i + 1].clone(), x[b, i].clone()
    return perm + noise, p, r


def packet_subsample(x, p, r, ratio=0.2):
    T = x.size(1)
    mask = (torch.rand(x.size(0), T, device=x.device) > ratio).float().unsqueeze(-1)
    return x * mask, p, r


def feature_masking(x, p, r, ratio=0.15):
    Fd = x.size(2)
    mask = (torch.rand(x.size(0), 1, Fd, device=x.device) > ratio).float()
    return x * mask, p, r


def augment(x, p, r):
    fns = [temporal_jitter, packet_subsample, feature_masking]
    a, b = random.sample(fns, 2)
    x1, p1, r1 = a(x, p, r)
    x2, p2, r2 = b(x, p, r)
    return (x1, p1, r1), (x2, p2, r2)


class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TransformerEncoderLayerAttn(nn.Module):
    def __init__(self, d, heads, ff, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff, d))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        a, w = self.attn(x, x, x, need_weights=True, average_attn_weights=True)
        x = self.norm1(x + self.drop(a))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x, w


class TransformerBranch(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.pe = PositionalEncoding(cfg.d_model)
        self.layers = nn.ModuleList(
            [TransformerEncoderLayerAttn(cfg.d_model, cfg.n_heads, cfg.ff_dim, cfg.dropout)
             for _ in range(cfg.n_transformer_layers)]
        )

    def forward(self, x):
        x = self.pe(x)
        attns = []
        for layer in self.layers:
            x, w = layer(x)
            attns.append(w)
        return x, attns


class BiLSTMBranch(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.lstm = nn.LSTM(cfg.d_model, cfg.lstm_hidden, num_layers=cfg.n_lstm_layers,
                            batch_first=True, bidirectional=True,
                            dropout=cfg.dropout if cfg.n_lstm_layers > 1 else 0.0)
        self.proj = nn.Linear(2 * cfg.lstm_hidden, cfg.d_model)

    def forward(self, x):
        h, _ = self.lstm(x)
        return self.proj(h)


class ThreatContextEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.port_emb = nn.Embedding(cfg.n_port_buckets, cfg.ctx_emb_dim)
        self.proto_emb = nn.Embedding(cfg.n_protocols, cfg.ctx_emb_dim)
        self.proj = nn.Sequential(
            nn.Linear(2 * cfg.ctx_emb_dim, cfg.d_model), nn.GELU(), nn.LayerNorm(cfg.d_model)
        )
        self.score = nn.Linear(cfg.d_model, 1)
        self.use_phi = cfg.use_phi

    def forward(self, port, proto):
        c = torch.cat([self.port_emb(port), self.proto_emb(proto)], dim=-1)
        c_pos = self.proj(c)
        phi = c_pos.mean(dim=1)
        ms = self.score(c_pos).squeeze(-1)
        if not self.use_phi:
            phi = torch.zeros_like(phi)
        return phi, c_pos, ms


class VectorAdaptiveGate(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        out = cfg.d_model if cfg.vector_gate else 1
        self.fc = nn.Linear(3 * cfg.d_model, out)
        self.vector = cfg.vector_gate

    def forward(self, ht, hl, phi):
        a = torch.sigmoid(self.fc(torch.cat([ht, hl, phi], dim=-1)))
        return a


class FeatureRefinement(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.fc = nn.Linear(d, d)
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.norm(x + self.drop(F.gelu(self.fc(x))))


class TARLNetV2(nn.Module):
    def __init__(self, cfg, raw_dim, num_known):
        super().__init__()
        self.cfg = cfg
        self.num_known = num_known
        self.embed = nn.Linear(raw_dim, cfg.d_model)
        self.embed_norm = nn.LayerNorm(cfg.d_model)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, cfg.d_model))
        self.context = ThreatContextEncoder(cfg)
        self.transformer = TransformerBranch(cfg) if cfg.use_transformer else None
        self.lstm = BiLSTMBranch(cfg) if cfg.use_lstm else None
        self.gate = VectorAdaptiveGate(cfg)

        self.mtm_head = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(),
                                      nn.Linear(cfg.d_model, raw_dim))
        self.proj_head = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(),
                                       nn.Linear(cfg.d_model, cfg.proj_dim))
        self.svdd = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model, bias=False),
                                  nn.LeakyReLU(0.1),
                                  nn.Linear(cfg.d_model, cfg.svdd_dim, bias=False))
        self.refine = FeatureRefinement(cfg.d_model, cfg.dropout)
        self.classifier = nn.Linear(cfg.d_model, num_known)
        self.register_buffer("centers", torch.zeros(num_known, cfg.svdd_dim))
        self.register_buffer("temperature", torch.ones(1))

    def embed_seq(self, x, mtm_mask=None):
        e = self.embed_norm(self.embed(x))
        if mtm_mask is not None:
            m = mtm_mask.unsqueeze(-1).float()
            e = e * (1 - m) + self.mask_token * m
        return e

    def backbone(self, e, phi):
        r = e + phi.unsqueeze(1) if self.cfg.use_phi else e
        attns = None
        if self.transformer is not None:
            ht_seq, attns = self.transformer(r)
        else:
            ht_seq = torch.zeros_like(r)
        hl_seq = self.lstm(r) if self.lstm is not None else torch.zeros_like(r)
        ht, hl = ht_seq.mean(1), hl_seq.mean(1)
        a = self.gate(ht, hl, phi)
        z = a * ht + (1 - a) * hl
        f_seq = a.unsqueeze(1) * ht_seq + (1 - a.unsqueeze(1)) * hl_seq
        return dict(z=z, f_seq=f_seq, ht=ht, hl=hl, alpha=a, attns=attns)

    def encode(self, x, port, proto, mtm_mask=None):
        phi, c_pos, ms = self.context(port, proto)
        e = self.embed_seq(x, mtm_mask)
        out = self.backbone(e, phi)
        out["phi"] = phi
        out["mask_score"] = ms
        return out

    def logits(self, z):
        return self.classifier(self.refine(z))

    def svdd_dist(self, z):
        g = self.svdd(z)
        d = torch.cdist(g, self.centers) ** 2
        return d, g


def build_mask(scores, ratio, guided_frac):
    N, T = scores.shape
    n_mask = max(1, int(round(ratio * T)))
    n_guided = int(round(guided_frac * n_mask))
    mask = torch.zeros(N, T, dtype=torch.bool, device=scores.device)
    for b in range(N):
        order = torch.argsort(scores[b], descending=True)
        guided = order[:n_guided]
        rest = order[n_guided:]
        rand = rest[torch.randperm(rest.numel(), device=scores.device)][: n_mask - n_guided]
        idx = torch.cat([guided, rand])
        mask[b, idx] = True
    return mask


def nt_xent(z1, z2, temp):
    N = z1.size(0)
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)
    sim = z @ z.t() / temp
    sim.fill_diagonal_(-1e9)
    targets = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)]).to(z.device)
    return F.cross_entropy(sim, targets)


def focal_loss(logits, target, gamma, weight=None):
    logp = F.log_softmax(logits, dim=1)
    p = logp.exp()
    ce = F.nll_loss(logp, target, weight=weight, reduction="none")
    pt = p.gather(1, target.unsqueeze(1)).squeeze(1)
    return ((1 - pt) ** gamma * ce).mean()


def loader(tensors, bs, shuffle):
    return DataLoader(TensorDataset(*tensors), batch_size=bs, shuffle=shuffle, drop_last=False)


def pretrain(model, train, cfg):
    if not cfg.use_ssl:
        return
    model.train()
    Xtr, Ptr, Rtr, _ = train
    dl = loader((Xtr, Ptr, Rtr), cfg.batch_size, True)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    for ep in range(cfg.pretrain_epochs):
        gf = min(1.0, ep / max(1, cfg.curriculum_warmup))
        tot = 0.0
        for x, p, r in dl:
            x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
            with torch.no_grad():
                _, _, ms = model.context(p, r)
            mask = build_mask(ms, cfg.mask_ratio, gf)
            out = model.encode(x, p, r, mtm_mask=mask)
            recon = model.mtm_head(out["f_seq"])
            m = mask.unsqueeze(-1).float()
            per = F.smooth_l1_loss(recon, x, reduction="none")
            l_mtm = (per * m).sum() / (m.sum().clamp_min(1.0) * x.size(-1))

            (x1, p1, r1), (x2, p2, r2) = augment(x, p, r)
            o1 = model.encode(x1, p1, r1)
            o2 = model.encode(x2, p2, r2)
            z1 = model.proj_head(o1["z"])
            z2 = model.proj_head(o2["z"])
            l_con = nt_xent(z1, z2, cfg.ntxent_temp)

            loss = cfg.lambda_mtm * l_mtm + cfg.lambda_contrast * l_con
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item() * x.size(0)
        print(f"[pretrain] epoch {ep + 1}/{cfg.pretrain_epochs} loss {tot / len(dl.dataset):.4f}")


@torch.no_grad()
def init_centers(model, train, label_map, cfg):
    model.eval()
    Xtr, Ptr, Rtr, Ytr = train
    dl = loader((Xtr, Ptr, Rtr, Ytr), cfg.batch_size, False)
    K = model.num_known
    sums = torch.zeros(K, cfg.svdd_dim, device=DEVICE)
    cnts = torch.zeros(K, device=DEVICE)
    for x, p, r, y in dl:
        x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
        ymap = torch.tensor([label_map[int(v)] for v in y], device=DEVICE)
        g = model.svdd(model.encode(x, p, r)["z"])
        for k in range(K):
            sel = ymap == k
            if sel.any():
                sums[k] += g[sel].sum(0)
                cnts[k] += sel.sum()
    centers = sums / cnts.clamp_min(1.0).unsqueeze(1)
    eps = 0.1
    centers[centers.abs() < eps] = eps
    model.centers.copy_(centers)


def train_supervised(model, train, label_map, cfg, ewc=None, replay=None):
    model.train()
    Xtr, Ptr, Rtr, Ytr = train
    ymap_all = torch.tensor([label_map[int(v)] for v in Ytr])
    counts = torch.bincount(ymap_all, minlength=model.num_known).float()
    weight = (counts.sum() / counts.clamp_min(1.0))
    weight = (weight / weight.sum() * model.num_known).to(DEVICE)
    dl = loader((Xtr, Ptr, Rtr, Ytr), cfg.batch_size, True)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    for ep in range(cfg.supervised_epochs):
        tot = 0.0
        for x, p, r, y in dl:
            x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
            ym = torch.tensor([label_map[int(v)] for v in y], device=DEVICE)
            if replay is not None and len(replay) > 0:
                rx, rp, rr, ry = replay.sample(cfg.replay_per_batch)
                x = torch.cat([x, rx.to(DEVICE)], 0)
                p = torch.cat([p, rp.to(DEVICE)], 0)
                r = torch.cat([r, rr.to(DEVICE)], 0)
                ym = torch.cat([ym, ry.to(DEVICE)], 0)
            out = model.encode(x, p, r)
            logits = model.logits(out["z"])
            l_cls = focal_loss(logits, ym, cfg.focal_gamma, weight)
            g = model.svdd(out["z"])
            c = model.centers[ym]
            l_svdd = ((g - c) ** 2).sum(1).mean()
            loss = l_cls + cfg.lambda_svdd * l_svdd
            if ewc is not None:
                loss = loss + cfg.ewc_lambda * ewc.penalty(model)
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item() * x.size(0)
        print(f"[supervised] epoch {ep + 1}/{cfg.supervised_epochs} loss {tot / len(dl.dataset):.4f}")


@torch.no_grad()
def gather_logits(model, tensors, label_map, cfg):
    model.eval()
    X, P, R, Y = tensors
    dl = loader((X, P, R, Y), cfg.batch_size, False)
    logits_all, dist_all, y_all, ymap_all = [], [], [], []
    for x, p, r, y in dl:
        x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
        out = model.encode(x, p, r)
        logits_all.append(model.logits(out["z"]).cpu())
        d, _ = model.svdd_dist(out["z"])
        dist_all.append(d.min(1).values.cpu())
        y_all.append(y)
        ymap_all.append(torch.tensor([label_map.get(int(v), -1) for v in y]))
    return (torch.cat(logits_all), torch.cat(dist_all),
            torch.cat(y_all), torch.cat(ymap_all))


def calibrate_temperature(model, val, label_map, cfg):
    logits, _, _, ymap = gather_logits(model, val, label_map, cfg)
    keep = ymap >= 0
    logits, ymap = logits[keep], ymap[keep]
    T = torch.nn.Parameter(torch.ones(1))
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=60)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / T.clamp_min(0.05), ymap)
        loss.backward()
        return loss

    opt.step(closure)
    model.temperature.copy_(T.detach().clamp_min(0.05).to(DEVICE))
    return float(model.temperature.item())


def compute_threshold(model, val, label_map, cfg):
    _, dist, _, ymap = gather_logits(model, val, label_map, cfg)
    known = dist[ymap >= 0].numpy()
    return float(np.quantile(known, cfg.svdd_quantile))


def expected_calibration_error(probs, labels, n_bins=15):
    conf, pred = probs.max(1)
    acc = (pred == labels).float()
    bins = torch.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.any():
            ece += (m.float().mean() * (acc[m].mean() - conf[m].mean()).abs()).item()
    return ece


def evaluate(model, test, label_map, tau, cfg, known_classes):
    logits, dist, y_raw, ymap = gather_logits(model, test, label_map, cfg)
    probs = F.softmax(logits / model.temperature.cpu(), dim=1)
    is_unknown_true = (ymap < 0).long()
    is_unknown_pred = (dist > tau).long()

    osr_auc = float("nan")
    if is_unknown_true.sum() > 0 and is_unknown_true.sum() < len(is_unknown_true):
        osr_auc = roc_auc_score(is_unknown_true.numpy(), dist.numpy())
    osr_acc = accuracy_score(is_unknown_true.numpy(), is_unknown_pred.numpy())

    known_mask = (ymap >= 0) & (dist <= tau)
    res = {}
    if known_mask.any():
        pred = probs[known_mask].argmax(1)
        true = ymap[known_mask]
        res["known_acc"] = accuracy_score(true.numpy(), pred.numpy())
        res["known_f1_macro"] = f1_score(true.numpy(), pred.numpy(), average="macro", zero_division=0)
        res["ece"] = expected_calibration_error(probs[known_mask], ymap[known_mask])
    res["osr_auroc"] = osr_auc
    res["osr_acc"] = osr_acc
    res["unknown_recall"] = recall_score(is_unknown_true.numpy(), is_unknown_pred.numpy(), zero_division=0)
    res["known_count"] = int((ymap >= 0).sum())
    res["unknown_count"] = int((ymap < 0).sum())
    return res


@torch.no_grad()
def attention_rollout(model, x, p, r):
    model.eval()
    out = model.encode(x, p, r)
    attns = out["attns"]
    if attns is None:
        return None
    T = attns[0].size(-1)
    eye = torch.eye(T, device=attns[0].device).unsqueeze(0)
    roll = eye.repeat(attns[0].size(0), 1, 1)
    for a in attns:
        a = a + eye
        a = a / a.sum(-1, keepdim=True)
        roll = a @ roll
    importance = roll.mean(1)
    return importance


def integrated_gradients(model, x, p, r, target, steps=24):
    model.eval()
    baseline = torch.zeros_like(x)
    total = torch.zeros_like(x)
    with torch.backends.cudnn.flags(enabled=False):
        for s in range(1, steps + 1):
            xi = (baseline + (s / steps) * (x - baseline)).clone().requires_grad_(True)
            out = model.encode(xi, p, r)
            logit = model.logits(out["z"])[torch.arange(x.size(0)), target]
            grad = torch.autograd.grad(logit.sum(), xi)[0]
            total = total + grad
    attr = (x - baseline) * total / steps
    return attr.detach()


@torch.no_grad()
def fidelity_scores(model, x, p, r, target, ranking, fraction=0.5):
    model.eval()
    Fd = x.size(2)
    k = max(1, int(fraction * Fd))
    base = F.softmax(model.logits(model.encode(x, p, r)["z"]), 1)[torch.arange(x.size(0)), target]
    xd = x.clone()
    top = ranking.argsort(dim=1, descending=True)[:, :k]
    for b in range(x.size(0)):
        xd[b, :, top[b]] = 0.0
    deleted = F.softmax(model.logits(model.encode(xd, p, r)["z"]), 1)[torch.arange(x.size(0)), target]
    xi = torch.zeros_like(x)
    for b in range(x.size(0)):
        xi[b, :, top[b]] = x[b, :, top[b]]
    inserted = F.softmax(model.logits(model.encode(xi, p, r)["z"]), 1)[torch.arange(x.size(0)), target]
    return float((base - deleted).mean()), float(inserted.mean())


class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.data = []

    def __len__(self):
        return len(self.data)

    def add(self, x, p, r, y):
        for i in range(x.size(0)):
            item = (x[i].cpu(), p[i].cpu(), r[i].cpu(), int(y[i]))
            if len(self.data) < self.capacity:
                self.data.append(item)
            else:
                self.data[random.randint(0, self.capacity - 1)] = item

    def sample(self, n):
        n = min(n, len(self.data))
        batch = random.sample(self.data, n)
        x = torch.stack([b[0] for b in batch])
        p = torch.stack([b[1] for b in batch])
        r = torch.stack([b[2] for b in batch])
        y = torch.tensor([b[3] for b in batch], dtype=torch.long)
        return x, p, r, y


class EWC:
    def __init__(self, model, tensors, label_map, cfg):
        self.params = {n: pr.detach().clone() for n, pr in model.named_parameters() if pr.requires_grad}
        self.fisher = {n: torch.zeros_like(pr) for n, pr in self.params.items()}
        model.eval()
        X, P, R, Y = tensors
        dl = loader((X, P, R, Y), cfg.batch_size, True)
        n = 0
        with torch.backends.cudnn.flags(enabled=False):
            for x, p, r, y in dl:
                x, p, r = x.to(DEVICE), p.to(DEVICE), r.to(DEVICE)
                ym = torch.tensor([label_map[int(v)] for v in y], device=DEVICE)
                model.zero_grad()
                logits = model.logits(model.encode(x, p, r)["z"])
                loss = F.cross_entropy(logits, ym)
                loss.backward()
                for nm, pr in model.named_parameters():
                    if pr.grad is not None and nm in self.fisher:
                        self.fisher[nm] += pr.grad.detach() ** 2 * x.size(0)
                n += x.size(0)
        for nm in self.fisher:
            self.fisher[nm] /= max(1, n)

    def penalty(self, model):
        loss = 0.0
        for nm, pr in model.named_parameters():
            if nm in self.fisher and pr.shape == self.params[nm].shape:
                loss = loss + (self.fisher[nm] * (pr - self.params[nm]) ** 2).sum()
        return loss


def known_label_map(known_classes):
    return {c: i for i, c in enumerate(sorted(known_classes))}


def filter_known(tensors, known_classes):
    X, P, R, Y = tensors
    keep = torch.tensor([int(v) in known_classes for v in Y])
    return X[keep], P[keep], R[keep], Y[keep]


def run_pipeline(cfg, train, val, test, known_classes):
    raw_dim = train[0].size(2)
    label_map = known_label_map(known_classes)
    model = TARLNetV2(cfg, raw_dim, len(known_classes)).to(DEVICE)

    train_k = filter_known(train, known_classes)
    val_k = filter_known(val, known_classes)

    pretrain(model, train_k, cfg)
    init_centers(model, train_k, label_map, cfg)
    train_supervised(model, train_k, label_map, cfg)
    T = calibrate_temperature(model, val_k, label_map, cfg)
    tau = compute_threshold(model, val_k, label_map, cfg)
    print(f"[calibration] temperature {T:.3f}  threshold tau {tau:.3f}")
    res = evaluate(model, test, label_map, tau, cfg, known_classes)
    return model, label_map, tau, res


def run_loafo(cfg, train, val, test, held_family):
    held = {i for i, c in enumerate(CLASS_NAMES) if ATTACK_FAMILY[c] == held_family}
    known = {i for i in range(len(CLASS_NAMES)) if i not in held}
    print(f"\n=== LOAFO fold: holding out family '{held_family}' "
          f"({[CLASS_NAMES[i] for i in held]}) ===")
    model, label_map, tau, res = run_pipeline(cfg, train, val, test, known)
    print("LOAFO result:", {k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()})
    return model, res


def run_ablation(cfg, train, val, test, held_family="recon"):
    held = {i for i, c in enumerate(CLASS_NAMES) if ATTACK_FAMILY[c] == held_family}
    known = {i for i in range(len(CLASS_NAMES)) if i not in held}
    variants = {
        "full": dict(),
        "no_transformer": dict(use_transformer=False),
        "no_lstm": dict(use_lstm=False),
        "no_phi": dict(use_phi=False),
        "no_ssl": dict(use_ssl=False),
        "scalar_gate": dict(vector_gate=False),
    }
    table = {}
    for name, overrides in variants.items():
        c = copy.deepcopy(cfg)
        for k, v in overrides.items():
            setattr(c, k, v)
        print(f"\n=== Ablation variant: {name} (open-set, holdout '{held_family}') ===")
        _, _, _, res = run_pipeline(c, train, val, test, known)
        table[name] = res
    print(f"\n=== Ablation summary (open-set, holdout '{held_family}') ===")
    for name, r in table.items():
        auroc = r.get("osr_auroc", float("nan"))
        auroc = f"{auroc:.4f}" if isinstance(auroc, float) and not math.isnan(auroc) else str(auroc)
        print(f"{name:16s} acc={r.get('known_acc', float('nan')):.4f} "
              f"f1={r.get('known_f1_macro', float('nan')):.4f} "
              f"osr_auroc={auroc} "
              f"unk_recall={r.get('unknown_recall', float('nan')):.4f}")
    return table


def demo_continual_learning(cfg, base_model, base_known, train, val, test):
    print("\n=== Continual learning demo (replay + EWC) ===")
    label_map = known_label_map(base_known)
    base_train = filter_known(train, base_known)
    ewc = EWC(base_model, base_train, label_map, cfg)
    replay = ReplayBuffer(cfg.replay_capacity)
    Xb, Pb, Rb, Yb = base_train
    yb_mapped = torch.tensor([label_map[int(v)] for v in Yb])
    replay.add(Xb[:cfg.replay_capacity], Pb[:cfg.replay_capacity],
               Rb[:cfg.replay_capacity], yb_mapped[:cfg.replay_capacity])

    missing = sorted(set(range(len(CLASS_NAMES))) - set(base_known))
    new_class = missing[0] if missing else None
    if new_class is None:
        print("no further class available for the demo")
        return base_model
    new_train = filter_known(train, {new_class})
    if new_train[0].shape[0] < 4:
        print(f"emerging class {CLASS_NAMES[new_class]} has too few samples; skipping")
        return base_model
    new_known = set(base_known) | {new_class}
    new_map = known_label_map(new_known)

    old_cls = base_model.classifier
    new_cls = nn.Linear(cfg.d_model, len(new_known)).to(DEVICE)
    with torch.no_grad():
        for c in base_known:
            new_cls.weight[new_map[c]] = old_cls.weight[label_map[c]]
            new_cls.bias[new_map[c]] = old_cls.bias[label_map[c]]
    base_model.classifier = new_cls
    base_model.num_known = len(new_known)
    new_centers = torch.zeros(len(new_known), cfg.svdd_dim, device=DEVICE)
    for c in base_known:
        new_centers[new_map[c]] = base_model.centers[label_map[c]]
    base_model.register_buffer("centers", new_centers)

    new_train = filter_known(train, {new_class})
    init_for_new = filter_known(train, new_known)
    init_centers(base_model, init_for_new, new_map, cfg)

    c = copy.deepcopy(cfg)
    c.supervised_epochs = max(4, cfg.supervised_epochs // 2)
    train_supervised(base_model, new_train, new_map, c, ewc=ewc, replay=replay)

    tau = compute_threshold(base_model, filter_known(val, new_known), new_map, cfg)
    res = evaluate(base_model, test, new_map, tau, cfg, new_known)
    print("after continual update:",
          {k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()})
    return base_model


def demo_xai(model, test, label_map, cfg):
    print("\n=== XAI demo on one batch ===")
    X, P, R, Y = test
    keep = torch.tensor([int(v) in label_map for v in Y])
    X, P, R, Y = X[keep][:8], P[keep][:8], R[keep][:8], Y[keep][:8]
    x, p, r = X.to(DEVICE), P.to(DEVICE), R.to(DEVICE)
    target = torch.tensor([label_map[int(v)] for v in Y], device=DEVICE)
    roll = attention_rollout(model, x, p, r)
    if roll is not None:
        print("attention rollout (per-timestep importance, sample 0):",
              np.round(roll[0].cpu().numpy(), 3))
    attr = integrated_gradients(model, x, p, r, target)
    feat_rank = attr.abs().mean(1)
    top = feat_rank[0].argsort(descending=True)[:5].cpu().numpy()
    print("top critical features (sample 0):", top.tolist())
    deletion, insertion = fidelity_scores(model, x, p, r, target, feat_rank)
    print(f"fidelity  deletion(drop)={deletion:.4f}  insertion(retain)={insertion:.4f}")


def main():
    cfg = Config()
    print("device:", DEVICE)
    if cfg.dataset == "unsw":
        train, val, test, feat_idx, scaler = make_splits_unsw(cfg)
        held = "reconnaissance"
    else:
        set_taxonomy(["Normal", "DoS", "DDoS", "Probe", "Botnet",
                      "BruteForce", "WebAttack", "Infiltration"],
                     {"Normal": "benign", "DoS": "volumetric", "DDoS": "volumetric",
                      "Probe": "recon", "Botnet": "c2", "BruteForce": "credential",
                      "WebAttack": "webapp", "Infiltration": "intrusion"})
        train, val, test, feat_idx, scaler = make_splits(cfg)
        held = "recon"

    print(f"dataset={cfg.dataset}  classes={CLASS_NAMES}")
    print(f"sequences  train={tuple(train[0].shape)}  val={tuple(val[0].shape)}  test={tuple(test[0].shape)}")
    print(f"selected features ({len(feat_idx)}):", feat_idx.tolist())
    print("train label distribution:", torch.bincount(train[3], minlength=len(CLASS_NAMES)).tolist())

    known = set(range(len(CLASS_NAMES)))
    model, label_map, tau, res = run_pipeline(cfg, train, val, test, known)
    print("\n=== Closed-set baseline (all classes known) ===")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()})

    demo_xai(model, test, label_map, cfg)
    run_loafo(cfg, train, val, test, held_family=held)

    base_known = set(range(len(CLASS_NAMES)))
    if cfg.dataset == "unsw" and "Fuzzers" in CLASS_NAMES:
        base_known = base_known - {CLASS_NAMES.index("Fuzzers")}
    else:
        base_known = set(range(len(CLASS_NAMES) - 1))
    base_model, base_map, base_tau, base_res = run_pipeline(cfg, train, val, test, base_known)
    demo_continual_learning(cfg, base_model, base_known, train, val, test)

    run_ablation(cfg, train, val, test, held_family=held)


if __name__ == "__main__":
    main()

device: cuda


100%|██████████| 149M/149M [00:04<00:00, 37.0MB/s]

Extracting files...


dataset=unsw  classes=['Normal', 'Fuzzers', 'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']
sequences  train=(19283, 16, 26)  val=(16000, 16, 26)  test=(16000, 16, 26)
selected features (26): [0, 1, 2, 3, 4, 5, 7, 8, 9, 11, 13, 14, 15, 16, 18, 19, 20, 21, 22, 24, 25, 27, 35, 36, 37, 38]
train label distribution: [12111, 468, 300, 300, 300, 867, 4037, 300, 300, 300]
[pretrain] epoch 1/6 loss 12.7687
[pretrain] epoch 2/6 loss 11.2584
[pretrain] epoch 3/6 loss 8.7750
[pretrain] epoch 4/6 loss 6.0090
[pretrain] epoch 5/6 loss 5.7522
[pretrain] epoch 6/6 loss 5.5858
[supervised] epoch 1/12 loss 0.7809
[supervised] epoch 2/12 loss 0.3670
[supervised] epoch 3/12 loss 0.3116
[supervised] epoch 4/12 loss 0.2971
[supervised] epoch 5/12 loss 0.2801
[supervised] epoch 6/12 loss 0.2654
[supervised] epoch 7/12 loss 0.2529
[supervised] epoch 8/12 loss 0.2411
[supervised] epoch 9/12 loss 0.2352
[supervised] epoch 10/12 loss 0.2271
[supervised] epoch 11/12 